# MathE Hierarchical Rasch Analysis with Posterior Predictive Validation

This notebook reproduces the full psychometric workflow for the MathE manuscript. It starts from the raw CSV and includes data-quality checks, repeated-attempt handling, item-exposure filtering, the final hierarchical Bayesian Rasch model, MCMC diagnostics, posterior item difficulty, Basic-hard/Advanced-easy mismatch flags, posterior predictive checks, exposure-threshold sensitivity, repeated-attempt sensitivity, prior sensitivity, and manuscript-ready exports.

The principal model is

\[
Y_{ij}\sim\mathrm{Bernoulli}(p_{ij}),\qquad
\mathrm{logit}(p_{ij})=\theta_i-b_j,
\]

with

\[
b_j=\mu_b+\beta_{\mathrm{level}}L_j+\gamma_{T(j)}+u_j.
\]

A positive \(\beta_{\mathrm{level}}\) means that Advanced-labelled items are more difficult than Basic-labelled items after accounting for student ability and mathematical topic.

## Environment setup
Install the packages in `requirements.txt` before running this notebook. The notebook was originally developed in Google Colab; a local Python environment is also supported.


In [ ]:
# 2. IMPORTS AND REPRODUCIBILITY
from pathlib import Path
import os, sys, json, random, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
warnings.filterwarnings("ignore")
SEED = 20260830
rng = np.random.default_rng(SEED)
random.seed(SEED)
np.random.seed(SEED)
print("Python:", sys.version)
print("PyMC:", pm.__version__)
print("ArviZ:", az.__version__)
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)

In [ ]:
# 3. PATHS
DATA_PATH = Path('data/MathE dataset (4).csv')
# Download the public CSV from the UCI MathE dataset and place it in data/.
# Alternatively, set DATA_PATH to your local copy.
OUTPUT_DIR = Path('mathe_final_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(DATA_PATH, OUTPUT_DIR.resolve())

In [ ]:
# 4. LOAD RAW DATA
raw = pd.read_csv(DATA_PATH, sep=';', encoding='cp1252')
print('Raw shape:', raw.shape)
display(raw.head())
print(raw.columns.tolist())

In [ ]:
# 5. STANDARDIZE COLUMNS AND RESPONSE CODING
rename_map = {
    'Student ID':'student_id','Student Country':'country','Question ID':'question_id',
    'Type of Answer':'correct','Question Level':'level','Topic':'topic',
    'Subtopic':'subtopic','Keywords':'keywords'
}
df = raw.rename(columns=rename_map).copy()
for c in ['student_id','question_id','country','level','topic','subtopic','keywords']:
    df[c] = df[c].astype(str).str.strip()
if not set(pd.Series(df['correct']).dropna().unique()).issubset({0,1}):
    mapping={'correct':1,'incorrect':0,'true':1,'false':0,'1':1,'0':0}
    df['correct'] = df['correct'].astype(str).str.strip().str.lower().map(mapping)
df['correct'] = pd.to_numeric(df['correct'], errors='coerce')
assert df['correct'].dropna().isin([0,1]).all()
print(df.shape)
display(df.head())

In [ ]:
# 6. DATA-QUALITY AUDIT
print('Missing values:')
display(df.isna().sum().to_frame('missing'))
print('Students:', df.student_id.nunique())
print('Questions:', df.question_id.nunique())
print('Countries:', df.country.nunique())
print('Correct:', int((df.correct==1).sum()))
print('Incorrect:', int((df.correct==0).sum()))
display(df.level.value_counts(dropna=False).to_frame('responses'))
display(df.topic.value_counts(dropna=False).to_frame('responses'))

In [ ]:
# 7. ITEM-METADATA CONSISTENCY CHECKS
rows=[]
for col in ['level','topic','subtopic','keywords']:
    counts=df.groupby('question_id')[col].nunique(dropna=False)
    for qid,n in counts[counts>1].items():
        vals=sorted(df.loc[df.question_id==qid,col].astype(str).unique().tolist())
        rows.append({'question_id':qid,'field':col,'n_values':int(n),'values':' | '.join(vals)})
metadata_issues=pd.DataFrame(rows)
print('Metadata inconsistencies:')
display(metadata_issues if len(metadata_issues) else pd.DataFrame({'status':['None']}))
metadata_issues.to_csv(OUTPUT_DIR/'metadata_consistency_issues.csv',index=False)

In [ ]:
# 8. REPEATED-ATTEMPT STRUCTURE
pair_counts=(df.groupby(['student_id','question_id']).size().rename('n_attempts').reset_index())
repeated_pairs=pair_counts.query('n_attempts>1').copy()
pair_outcome_nunique=(df.groupby(['student_id','question_id']).correct.nunique().rename('n_outcomes').reset_index())
repeated_detail=repeated_pairs.merge(pair_outcome_nunique,on=['student_id','question_id'])
n_unique_pairs=len(pair_counts)
n_repeated_pairs=len(repeated_pairs)
n_conflicting_pairs=int((repeated_detail.n_outcomes>1).sum())
print('Raw rows:',len(df))
print('Unique student-question pairs:',n_unique_pairs)
print('Repeated pairs:',n_repeated_pairs)
print('Repeated pairs with conflicting outcomes:',n_conflicting_pairs)
print('Rows belonging to repeated pairs:',int(repeated_pairs.n_attempts.sum()))

In [ ]:
# 9. PRIMARY SINGLE-ATTEMPT SAMPLE
work=df.merge(pair_counts,on=['student_id','question_id'],how='left')
single=work.loc[work.n_attempts==1].copy()
print('Single-attempt rows:',len(single))
print('Students:',single.student_id.nunique())
print('Questions:',single.question_id.nunique())

In [ ]:
# 10. ITEM-EXPOSURE TABLE
threshold_rows=[]
for t in [3,5,10,15,20,30]:
    counts=single.groupby('question_id').size()
    keep=counts[counts>=t].index
    tmp=single[single.question_id.isin(keep)]
    ilevel=tmp.groupby('question_id').level.first()
    threshold_rows.append({'threshold':t,'responses':len(tmp),'items':tmp.question_id.nunique(),
        'basic_items':int((ilevel=='Basic').sum()),'advanced_items':int((ilevel=='Advanced').sum())})
threshold_table=pd.DataFrame(threshold_rows)
display(threshold_table)
threshold_table.to_csv(OUTPUT_DIR/'item_exposure_thresholds.csv',index=False)

In [ ]:
# 11. PRIMARY ANALYTICAL SAMPLE: >=10 INDEPENDENT RESPONSES PER ITEM
PRIMARY_THRESHOLD=10
item_counts=single.groupby('question_id').size()
keep_items=item_counts[item_counts>=PRIMARY_THRESHOLD].index
primary=single[single.question_id.isin(keep_items)].copy()
ilevel=primary.groupby('question_id').level.first()
print('Primary responses:',len(primary))
print('Students:',primary.student_id.nunique())
print('Items:',primary.question_id.nunique())
print('Basic items:',int((ilevel=='Basic').sum()))
print('Advanced items:',int((ilevel=='Advanced').sum()))

In [ ]:
# 12. TOPIC SUPPORT AND DEFINITIVE MCMC SAMPLE
# Exclude topic groups represented by fewer than 2 retained items from topic-comparison MCMC.
topic_item_counts=primary.groupby('topic').question_id.nunique().sort_values(ascending=False)
display(topic_item_counts.to_frame('items'))
valid_topics=topic_item_counts[topic_item_counts>=2].index.tolist()
mcmc_df=primary[primary.topic.isin(valid_topics)].copy()
print('MCMC responses:',len(mcmc_df))
print('MCMC items:',mcmc_df.question_id.nunique())
print('Topics:',sorted(mcmc_df.topic.unique()))

In [ ]:
# 13. DESCRIPTIVE PERFORMANCE
raw_level=single.groupby('level').correct.agg(['mean','count']).rename(columns={'mean':'proportion_correct','count':'responses'})
primary_level=primary.groupby('level').correct.agg(['mean','count']).rename(columns={'mean':'proportion_correct','count':'responses'})
print('Single-attempt sample:')
display(raw_level)
print('Primary sample:')
display(primary_level)
raw_level.to_csv(OUTPUT_DIR/'raw_accuracy_by_level_single_attempt.csv')
primary_level.to_csv(OUTPUT_DIR/'raw_accuracy_by_level_primary.csv')

In [ ]:
# 14. RAW ITEM-SUCCESS FIGURE
item_raw=primary.groupby(['question_id','level','topic'],as_index=False).agg(n=('correct','size'),proportion_correct=('correct','mean'))
fig,ax=plt.subplots(figsize=(8,5))
levels=['Basic','Advanced']; positions=[1,2]
data=[item_raw.loc[item_raw.level==lev,'proportion_correct'].values for lev in levels]
ax.boxplot(data,positions=positions,widths=.55)
for x,lev in zip(positions,levels):
    yy=item_raw.loc[item_raw.level==lev,'proportion_correct'].values
    ax.scatter(rng.normal(x,.045,len(yy)),yy,alpha=.45,s=24)
ax.set_xticks(positions); ax.set_xticklabels(levels)
ax.set_ylabel('Observed item proportion correct'); ax.set_xlabel('Expert-assigned level')
ax.set_title('Observed item success rates by assigned level')
fig.tight_layout(); fig.savefig(OUTPUT_DIR/'Fig_raw_item_success_by_level.png',dpi=300,bbox_inches='tight'); plt.show()

In [ ]:
# 15. ENCODE INDICES FOR HIERARCHICAL RASCH MODEL
model_df=mcmc_df.copy()
students=sorted(model_df.student_id.unique())
items=sorted(model_df.question_id.unique())
reference_topic='Linear Algebra'
all_topics=sorted(model_df.topic.unique())
assert reference_topic in all_topics
nonreference_topics=[t for t in all_topics if t!=reference_topic]
student_to_idx={s:i for i,s in enumerate(students)}
item_to_idx={q:i for i,q in enumerate(items)}
topic_to_idx={t:i for i,t in enumerate(nonreference_topics)}
model_df['student_idx']=model_df.student_id.map(student_to_idx).astype(int)
model_df['item_idx']=model_df.question_id.map(item_to_idx).astype(int)
model_df['advanced']=(model_df.level=='Advanced').astype(int)
item_meta=(model_df.sort_values('question_id').groupby('question_id',as_index=False).first()[['question_id','level','topic']])
item_meta['item_idx']=item_meta.question_id.map(item_to_idx).astype(int)
item_meta['advanced']=(item_meta.level=='Advanced').astype(int)
X_topic=np.zeros((len(items),len(nonreference_topics)))
advanced_item=np.zeros(len(items))
for _,r in item_meta.iterrows():
    advanced_item[r.item_idx]=r.advanced
    if r.topic!=reference_topic: X_topic[r.item_idx,topic_to_idx[r.topic]]=1.0
y=model_df.correct.astype(int).to_numpy()
student_idx=model_df.student_idx.to_numpy(); item_idx=model_df.item_idx.to_numpy()
coords={'student':students,'item':items,'topic_effect':nonreference_topics,'obs':np.arange(len(model_df))}
print(len(students),len(items),nonreference_topics,len(y))

In [ ]:
# 16. FINAL HIERARCHICAL BAYESIAN RASCH MODEL
with pm.Model(coords=coords) as rasch_model:
    student_index=pm.Data('student_index',student_idx,dims='obs')
    item_index=pm.Data('item_index',item_idx,dims='obs')
    advanced_item_data=pm.Data('advanced_item_data',advanced_item,dims='item')
    X_topic_data=pm.Data('X_topic_data',X_topic,dims=('item','topic_effect'))
    mu_b=pm.Normal('mu_b',0,1.5)
    beta_level=pm.Normal('beta_level',0,1.5)
    gamma_topic=pm.Normal('gamma_topic',0,1.5,dims='topic_effect')
    sigma_theta=pm.HalfNormal('sigma_theta',1.0)
    theta_raw=pm.Normal('theta_raw',0,1,dims='student')
    theta=pm.Deterministic('theta',theta_raw*sigma_theta,dims='student')
    sigma_item=pm.HalfNormal('sigma_item',1.0)
    u_raw=pm.Normal('u_raw',0,1,dims='item')
    u_item=pm.Deterministic('u_item',u_raw*sigma_item,dims='item')
    topic_component=pm.math.dot(X_topic_data,gamma_topic)
    b_item=pm.Deterministic('b_item',mu_b+beta_level*advanced_item_data+topic_component+u_item,dims='item')
    eta=theta[student_index]-b_item[item_index]
    p=pm.Deterministic('p',pm.math.sigmoid(eta),dims='obs')
    pm.Bernoulli('y_obs',p=p,observed=y,dims='obs')
rasch_model

In [ ]:
# 17. SAMPLE THE POSTERIOR
with rasch_model:
    idata=pm.sample(draws=2000,tune=2000,chains=4,cores=4,target_accept=.95,
        random_seed=[101,202,303,404],return_inferencedata=True,
        idata_kwargs={'log_likelihood':True})
print(idata)

In [ ]:
# 18. MCMC CONVERGENCE DIAGNOSTICS
summary_vars=['mu_b','beta_level','sigma_theta','sigma_item','gamma_topic']
summary=az.summary(idata,var_names=summary_vars,kind='stats',ci_prob=.95)
diag=az.summary(idata,var_names=summary_vars,ci_prob=.95)
display(summary); display(diag)
summary.to_csv(OUTPUT_DIR/'MCMC_parameter_summary.csv'); diag.to_csv(OUTPUT_DIR/'MCMC_full_diagnostics.csv')
print('Maximum R-hat:',float(diag.r_hat.max()))
print('Minimum bulk ESS:',float(diag.ess_bulk.min()))
print('Minimum tail ESS:',float(diag.ess_tail.min()))

In [ ]:
# 19. DIVERGENT TRANSITIONS
divergences=int(idata.sample_stats['diverging'].sum().values)
total_draws=int(np.prod(idata.sample_stats['diverging'].shape))
print('Divergences:',divergences,'of',total_draws)
pd.DataFrame({'divergences':[divergences],'total_postwarmup_chain_draws':[total_draws],
    'divergence_proportion':[divergences/total_draws]}).to_csv(OUTPUT_DIR/'MCMC_divergence_summary.csv',index=False)

In [ ]:
# 20. TRACE PLOTS
az.plot_trace(idata,var_names=['beta_level','sigma_theta','sigma_item'],compact=False)
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'Fig_MCMC_trace.png',dpi=300,bbox_inches='tight'); plt.show()

In [ ]:
# 21. ADVANCED-LEVEL POSTERIOR
beta=idata.posterior['beta_level'].stack(sample=('chain','draw')).values
beta_result=pd.DataFrame({'posterior_mean':[beta.mean()],'posterior_median':[np.median(beta)],
    'q025':[np.quantile(beta,.025)],'q975':[np.quantile(beta,.975)],
    'P_beta_gt_0':[(beta>0).mean()],'P_beta_lt_0':[(beta<0).mean()]})
display(beta_result); beta_result.to_csv(OUTPUT_DIR/'MCMC_beta_level_result.csv',index=False)
fig,ax=plt.subplots(figsize=(8,5)); ax.hist(beta,bins=60,density=True,alpha=.7)
ax.axvline(0,ls='--'); ax.axvline(np.median(beta))
ax.set_xlabel(r'$\beta_{\mathrm{level}}$'); ax.set_ylabel('Posterior density')
ax.set_title('Posterior distribution of Advanced-item difficulty effect')
fig.tight_layout(); fig.savefig(OUTPUT_DIR/'Fig_beta_level_posterior.png',dpi=300,bbox_inches='tight'); plt.show()

In [ ]:
# 22. POSTERIOR ITEM DIFFICULTIES
b=idata.posterior['b_item'].stack(sample=('chain','draw')).transpose('item','sample').values
item_results=item_meta.copy()
item_results['b_mean']=b.mean(axis=1); item_results['b_median']=np.median(b,axis=1)
item_results['b_q025']=np.quantile(b,.025,axis=1); item_results['b_q975']=np.quantile(b,.975,axis=1)
raw_item_stats=model_df.groupby('question_id').correct.agg(['mean','count']).rename(columns={'mean':'raw_prop_correct','count':'n_responses'}).reset_index()
item_results=item_results.merge(raw_item_stats,on='question_id',how='left')
display(item_results.sort_values('b_median',ascending=False).head(15))
item_results.to_csv(OUTPUT_DIR/'MCMC_item_difficulty.csv',index=False)

In [ ]:
# 23. BASIC-HARD / ADVANCED-EASY FLAGS
q25=item_results.b_median.quantile(.25); q75=item_results.b_median.quantile(.75)
def flag(r):
    if r.level=='Basic' and r.b_median>=q75: return 'Basic-hard'
    if r.level=='Advanced' and r.b_median<=q25: return 'Advanced-easy'
    return 'Neither'
item_results['mismatch_flag']=item_results.apply(flag,axis=1)
print('q25=',q25,'q75=',q75); display(item_results.mismatch_flag.value_counts().to_frame('items'))
item_results.to_csv(OUTPUT_DIR/'MCMC_item_difficulty_with_mismatch.csv',index=False)

In [ ]:
# 24. TOPIC-EFFECT POSTERIOR SUMMARY
gamma=idata.posterior['gamma_topic'].stack(sample=('chain','draw')).transpose('topic_effect','sample').values
rows=[{'topic':reference_topic,'posterior_mean':0.0,'posterior_median':0.0,'q025':0.0,'q975':0.0,'reference':True}]
for i,t in enumerate(nonreference_topics):
    v=gamma[i]; rows.append({'topic':t,'posterior_mean':v.mean(),'posterior_median':np.median(v),
        'q025':np.quantile(v,.025),'q975':np.quantile(v,.975),'reference':False})
topic_post=pd.DataFrame(rows); display(topic_post); topic_post.to_csv(OUTPUT_DIR/'MCMC_topic_effects.csv',index=False)

## Posterior predictive validation
Posterior predictive checking evaluates whether the fitted model can reproduce important features of the observed response data. It is distinct from MCMC convergence. We evaluate overall accuracy, Basic/Advanced accuracy, topic accuracy, item-level success, and student-level success.

In [ ]:
# 25. DRAW POSTERIOR PREDICTIVE REPLICATES
with rasch_model:
    ppc=pm.sample_posterior_predictive(idata,var_names=['y_obs'],random_seed=SEED,return_inferencedata=True)
idata.extend(ppc)
yrep=idata.posterior_predictive['y_obs'].stack(pp_sample=('chain','draw')).transpose('pp_sample','obs').values
yobs=y.astype(int)
print('PPC matrix:',yrep.shape)

In [ ]:
# 26. PPC: OVERALL ACCURACY
obs_overall=yobs.mean(); rep_overall=yrep.mean(axis=1)
overall_ppc=pd.DataFrame({'statistic':['overall_accuracy'],'observed':[obs_overall],
    'pp_mean':[rep_overall.mean()],'pp_median':[np.median(rep_overall)],
    'pp_q025':[np.quantile(rep_overall,.025)],'pp_q975':[np.quantile(rep_overall,.975)],
    'bayesian_p_ge_obs':[(rep_overall>=obs_overall).mean()]})
display(overall_ppc); overall_ppc.to_csv(OUTPUT_DIR/'PPC_overall_accuracy.csv',index=False)

In [ ]:
# 27. PPC: BASIC VS ADVANCED
rows=[]
for lev in ['Basic','Advanced']:
    mask=model_df.level.eq(lev).to_numpy(); obs=yobs[mask].mean(); rep=yrep[:,mask].mean(axis=1)
    rows.append({'level':lev,'n_observations':int(mask.sum()),'observed':obs,'pp_mean':rep.mean(),
        'pp_median':np.median(rep),'pp_q025':np.quantile(rep,.025),'pp_q975':np.quantile(rep,.975),
        'bayesian_p_ge_obs':(rep>=obs).mean()})
level_ppc=pd.DataFrame(rows); display(level_ppc); level_ppc.to_csv(OUTPUT_DIR/'PPC_accuracy_by_level.csv',index=False)

In [ ]:
# 28. PPC: TOPIC-SPECIFIC ACCURACY
rows=[]
for topic in sorted(model_df.topic.unique()):
    mask=model_df.topic.eq(topic).to_numpy(); obs=yobs[mask].mean(); rep=yrep[:,mask].mean(axis=1)
    rows.append({'topic':topic,'n_observations':int(mask.sum()),'observed':obs,'pp_mean':rep.mean(),
        'pp_median':np.median(rep),'pp_q025':np.quantile(rep,.025),'pp_q975':np.quantile(rep,.975),
        'bayesian_p_ge_obs':(rep>=obs).mean()})
topic_ppc=pd.DataFrame(rows); display(topic_ppc); topic_ppc.to_csv(OUTPUT_DIR/'PPC_accuracy_by_topic.csv',index=False)

In [ ]:
# 29. PPC: ITEM-LEVEL SUCCESS RATES
obs_item=model_df.groupby('item_idx').correct.mean().reindex(range(len(items))).to_numpy()
rep_item=np.empty((yrep.shape[0],len(items)))
for j in range(len(items)):
    mask=item_idx==j; rep_item[:,j]=yrep[:,mask].mean(axis=1)
item_ppc=item_meta.copy(); item_ppc['observed_success']=obs_item; item_ppc['pp_mean']=rep_item.mean(axis=0)
item_ppc['pp_median']=np.median(rep_item,axis=0); item_ppc['pp_q025']=np.quantile(rep_item,.025,axis=0); item_ppc['pp_q975']=np.quantile(rep_item,.975,axis=0)
item_ppc['inside_95_interval']=(item_ppc.observed_success>=item_ppc.pp_q025)&(item_ppc.observed_success<=item_ppc.pp_q975)
print('Item PPC coverage:',item_ppc.inside_95_interval.mean()); display(item_ppc.head())
item_ppc.to_csv(OUTPUT_DIR/'PPC_item_success_rates.csv',index=False)

In [ ]:
# 30. PPC FIGURE: OBSERVED VS PREDICTED ITEM SUCCESS
fig,ax=plt.subplots(figsize=(7,7)); ax.scatter(item_ppc.observed_success,item_ppc.pp_mean,alpha=.65,s=32)
ax.plot([0,1],[0,1],ls='--'); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_xlabel('Observed item proportion correct'); ax.set_ylabel('Posterior-predictive mean proportion correct')
ax.set_title('Posterior predictive check: item success rates')
fig.tight_layout(); fig.savefig(OUTPUT_DIR/'Fig_PPC_observed_vs_predicted_item_success.png',dpi=300,bbox_inches='tight'); plt.show()

In [ ]:
# 31. PPC FIGURE: ASSIGNED LEVEL
fig,ax=plt.subplots(figsize=(7,5)); x=np.arange(len(level_ppc))
ax.errorbar(x,level_ppc.pp_mean,yerr=[level_ppc.pp_mean-level_ppc.pp_q025,level_ppc.pp_q975-level_ppc.pp_mean],fmt='o',capsize=4,label='Posterior predictive')
ax.scatter(x,level_ppc.observed,marker='x',s=80,label='Observed'); ax.set_xticks(x); ax.set_xticklabels(level_ppc.level)
ax.set_ylabel('Proportion correct'); ax.set_ylim(0,1); ax.set_title('Posterior predictive check by assigned level'); ax.legend()
fig.tight_layout(); fig.savefig(OUTPUT_DIR/'Fig_PPC_accuracy_by_level.png',dpi=300,bbox_inches='tight'); plt.show()

In [ ]:
# 32. PPC FIGURE: TOPIC
plot_df=topic_ppc.sort_values('observed').reset_index(drop=True)
fig,ax=plt.subplots(figsize=(9,5)); x=np.arange(len(plot_df))
ax.errorbar(x,plot_df.pp_mean,yerr=[plot_df.pp_mean-plot_df.pp_q025,plot_df.pp_q975-plot_df.pp_mean],fmt='o',capsize=4,label='Posterior predictive')
ax.scatter(x,plot_df.observed,marker='x',s=80,label='Observed'); ax.set_xticks(x); ax.set_xticklabels(plot_df.topic,rotation=35,ha='right')
ax.set_ylabel('Proportion correct'); ax.set_ylim(0,1); ax.set_title('Posterior predictive check by mathematical topic'); ax.legend()
fig.tight_layout(); fig.savefig(OUTPUT_DIR/'Fig_PPC_accuracy_by_topic.png',dpi=300,bbox_inches='tight'); plt.show()

In [ ]:
# 33. PPC: STUDENT-LEVEL SUCCESS
obs_student=model_df.groupby('student_idx').correct.mean().reindex(range(len(students))).to_numpy()
rep_student=np.empty((yrep.shape[0],len(students)))
for i in range(len(students)):
    mask=student_idx==i; rep_student[:,i]=yrep[:,mask].mean(axis=1)
student_ppc=pd.DataFrame({'student_id':students,'observed_success':obs_student,'pp_mean':rep_student.mean(axis=0),
    'pp_q025':np.quantile(rep_student,.025,axis=0),'pp_q975':np.quantile(rep_student,.975,axis=0)})
student_ppc['inside_95_interval']=(student_ppc.observed_success>=student_ppc.pp_q025)&(student_ppc.observed_success<=student_ppc.pp_q975)
print('Student PPC coverage:',student_ppc.inside_95_interval.mean())
student_ppc.to_csv(OUTPUT_DIR/'PPC_student_success_rates.csv',index=False)

In [ ]:
# 34. GLOBAL PPC SUMMARY
ppc_summary=pd.DataFrame({'metric':['overall_accuracy','item_success_inside_95_interval','student_success_inside_95_interval','basic_accuracy','advanced_accuracy'],
'value':[obs_overall,item_ppc.inside_95_interval.mean(),student_ppc.inside_95_interval.mean(),
float(level_ppc.loc[level_ppc.level=='Basic','observed'].iloc[0]),float(level_ppc.loc[level_ppc.level=='Advanced','observed'].iloc[0])]})
display(ppc_summary); ppc_summary.to_csv(OUTPUT_DIR/'PPC_global_summary.csv',index=False)

## Sensitivity analyses
We assess whether the central conclusion depends strongly on item-exposure threshold, treatment of repeated attempts, or prior scale.

In [ ]:
# 35. EXPOSURE-THRESHOLD SENSITIVITY
rows=[]
for threshold in [5,10,15,20]:
    counts=single.groupby('question_id').size(); keep=counts[counts>=threshold].index
    tmp=single[single.question_id.isin(keep)]
    stats=tmp.groupby(['question_id','level'],as_index=False).agg(prop_correct=('correct','mean'))
    stats['raw_difficulty']=1-stats.prop_correct; means=stats.groupby('level').raw_difficulty.mean()
    b=means.get('Basic',np.nan); a=means.get('Advanced',np.nan)
    rows.append({'threshold':threshold,'basic_mean_raw_difficulty':b,'advanced_mean_raw_difficulty':a,
                 'advanced_minus_basic':a-b,'n_items':stats.question_id.nunique()})
threshold_sensitivity=pd.DataFrame(rows); display(threshold_sensitivity)
threshold_sensitivity.to_csv(OUTPUT_DIR/'sensitivity_item_exposure_raw_difficulty.csv',index=False)

In [ ]:
# 36. REPEATED-ATTEMPT SENSITIVITY
N_REPS=500; contrasts=[]
groups={k:g.copy() for k,g in df.groupby(['student_id','question_id'],sort=False)}
for _ in range(N_REPS):
    chosen=[]
    for _,g in groups.items():
        chosen.append(g.iloc[0] if len(g)==1 else g.iloc[rng.integers(0,len(g))])
    sampled=pd.DataFrame(chosen); means=sampled.groupby('level').correct.mean()
    contrasts.append(means.get('Advanced',np.nan)-means.get('Basic',np.nan))
repeat_sensitivity=pd.DataFrame({'replicate':np.arange(1,N_REPS+1),'advanced_minus_basic_raw_accuracy':contrasts})
display(repeat_sensitivity.describe()); repeat_sensitivity.to_csv(OUTPUT_DIR/'sensitivity_repeated_attempts.csv',index=False)
fig,ax=plt.subplots(figsize=(8,5)); ax.hist(repeat_sensitivity.advanced_minus_basic_raw_accuracy,bins=35); ax.axvline(0,ls='--')
ax.set_xlabel('Advanced minus Basic raw accuracy'); ax.set_ylabel('Replicates'); ax.set_title('Sensitivity to repeated-response selection')
fig.tight_layout(); fig.savefig(OUTPUT_DIR/'Fig_repeated_attempt_sensitivity.png',dpi=300,bbox_inches='tight'); plt.show()

## Prior sensitivity
The main model uses Normal(0,1.5) priors for the intercept, level effect, and topic effects. We refit with Normal(0,1.0) and Normal(0,2.5) priors. The central question is whether the posterior conclusion for \(eta_{\mathrm{level}}\) materially changes.

In [ ]:
# 37. PRIOR-SENSITIVITY MODEL FUNCTION
def fit_rasch_with_prior_scale(prior_scale,draws=1500,tune=1500):
    with pm.Model(coords=coords) as model:
        si=pm.Data('student_index',student_idx,dims='obs'); ii=pm.Data('item_index',item_idx,dims='obs')
        ad=pm.Data('advanced_item_data',advanced_item,dims='item'); xt=pm.Data('X_topic_data',X_topic,dims=('item','topic_effect'))
        mu=pm.Normal('mu_b',0,prior_scale); beta=pm.Normal('beta_level',0,prior_scale)
        gam=pm.Normal('gamma_topic',0,prior_scale,dims='topic_effect')
        st=pm.HalfNormal('sigma_theta',1); tr=pm.Normal('theta_raw',0,1,dims='student'); th=pm.Deterministic('theta',tr*st,dims='student')
        sb=pm.HalfNormal('sigma_item',1); ur=pm.Normal('u_raw',0,1,dims='item'); ui=pm.Deterministic('u_item',ur*sb,dims='item')
        bi=pm.Deterministic('b_item',mu+beta*ad+pm.math.dot(xt,gam)+ui,dims='item')
        pm.Bernoulli('y_obs',p=pm.math.sigmoid(th[si]-bi[ii]),observed=y,dims='obs')
        return pm.sample(draws=draws,tune=tune,chains=4,cores=4,target_accept=.95,
                         random_seed=[111,222,333,444],return_inferencedata=True)

In [ ]:
# 38. FIT PRIOR-SENSITIVITY MODELS
idata_prior_1=fit_rasch_with_prior_scale(1.0)
idata_prior_25=fit_rasch_with_prior_scale(2.5)

In [ ]:
# 39. PRIOR-SENSITIVITY SUMMARY
def summarize_beta(obj,label):
    v=obj.posterior['beta_level'].stack(sample=('chain','draw')).values
    return {'model':label,'mean':v.mean(),'median':np.median(v),'q025':np.quantile(v,.025),'q975':np.quantile(v,.975),
            'P_beta_gt_0':(v>0).mean(),'P_beta_lt_0':(v<0).mean()}
prior_sensitivity=pd.DataFrame([summarize_beta(idata,'Main: Normal(0,1.5)'),summarize_beta(idata_prior_1,'Narrower: Normal(0,1.0)'),summarize_beta(idata_prior_25,'Wider: Normal(0,2.5)')])
display(prior_sensitivity); prior_sensitivity.to_csv(OUTPUT_DIR/'sensitivity_prior_beta_level.csv',index=False)
fig,ax=plt.subplots(figsize=(9,5)); x=np.arange(len(prior_sensitivity))
ax.errorbar(x,prior_sensitivity['median'],yerr=[prior_sensitivity['median']-prior_sensitivity.q025,prior_sensitivity.q975-prior_sensitivity['median']],fmt='o',capsize=5)
ax.axhline(0,ls='--'); ax.set_xticks(x); ax.set_xticklabels(prior_sensitivity.model,rotation=20,ha='right')
ax.set_ylabel(r'Posterior $\beta_{\mathrm{level}}$'); ax.set_title('Prior sensitivity of Advanced-item difficulty effect')
fig.tight_layout(); fig.savefig(OUTPUT_DIR/'Fig_prior_sensitivity_beta_level.png',dpi=300,bbox_inches='tight'); plt.show()

In [ ]:
# 40. SOFTWARE VERSIONS AND SAVE INFERENCEDATA
versions=pd.DataFrame({'software':['Python','PyMC','ArviZ','NumPy','pandas'],
    'version':[platform.python_version(),pm.__version__,az.__version__,np.__version__,pd.__version__]})
display(versions); versions.to_csv(OUTPUT_DIR/'software_versions.csv',index=False)
idata.to_netcdf(OUTPUT_DIR/'MathE_final_hierarchical_rasch_idata.nc')

In [ ]:
# 41. MANUSCRIPT-READY RESULTS SUMMARY
beta_row=beta_result.iloc[0]
results_summary={'n_raw_rows':int(len(df)),'n_students_raw':int(df.student_id.nunique()),'n_items_raw':int(df.question_id.nunique()),
'n_unique_student_item_pairs':int(n_unique_pairs),'n_repeated_pairs':int(n_repeated_pairs),'n_conflicting_repeated_pairs':int(n_conflicting_pairs),
'n_single_attempt_rows':int(len(single)),'n_primary_rows':int(len(primary)),'n_primary_items':int(primary.question_id.nunique()),
'n_mcmc_items':int(model_df.question_id.nunique()),'beta_level_mean':float(beta_row.posterior_mean),
'beta_level_median':float(beta_row.posterior_median),'beta_level_q025':float(beta_row.q025),'beta_level_q975':float(beta_row.q975),
'P_beta_gt_0':float(beta_row.P_beta_gt_0),'P_beta_lt_0':float(beta_row.P_beta_lt_0),'max_rhat':float(diag.r_hat.max()),
'min_bulk_ess':float(diag.ess_bulk.min()),'min_tail_ess':float(diag.ess_tail.min()),'divergences':int(divergences),
'ppc_item_coverage_95':float(item_ppc.inside_95_interval.mean()),'ppc_student_coverage_95':float(student_ppc.inside_95_interval.mean())}
with open(OUTPUT_DIR/'manuscript_results_summary.json','w') as f: json.dump(results_summary,f,indent=2)
print(json.dumps(results_summary,indent=2))

In [ ]:
# 42. LIST ALL EXPORTED FILES
for p in sorted(OUTPUT_DIR.iterdir()): print(p.name)

# Interpretation checklist

Before updating the manuscript, verify:

- maximum \(\widehat R < 1.01\);
- adequate bulk and tail ESS;
- exact divergence count recorded;
- trace plots show satisfactory mixing;
- posterior predictive checks reproduce overall accuracy and broad level/topic patterns;
- no systematic item-level PPC failure;
- the 95% credible interval for \(\beta_{\mathrm{level}}\) remains compatible with little or no level effect;
- alternative prior scales do not materially change that conclusion;
- repeated-attempt and exposure-threshold sensitivity do not explain the central pattern.

Do not describe posterior predictive checks as satisfactory until the exported results support that statement.